# 🌱 FarmFusion Crop Disease Detection Model V2 (38-Class EfficientNet-B3)

**Project**: FarmFusion Multilingual AI Agricultural Copilot  
**Model**: EfficientNet-B3 Transfer Learning with Mixed Precision (AMP) & Probability Calibration  
**Dataset**: Full PlantVillage Dataset (~54,300 images across 38 crop & disease classes)  
**Target Hardware**: Google Colab GPU (NVIDIA Tesla T4 16GB)  
**Output Target**: `/content/exported_models_38/`  
**Safety Rules**: Pre-flight verification, stratified 80/10/10 split, confidence tiering, no hallucinated classes.

---

In [10]:
import os
os._exit(0)


: 

: 

: 

## 1. Environment Setup & Tesla T4 GPU Verification

In [1]:
!pip install -q timm torchvision torch scikit-learn matplotlib seaborn pandas numpy Pillow tqdm joblib

import os
import gc
import sys
import json
import time
import copy
import shutil
import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

# Prevent CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as transforms
import timm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, top_k_accuracy_score,
    confusion_matrix, classification_report
)

# Clear lingering GPU cache from previous runs
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

# Set seed for strict reproducibility
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=== RUNTIME DIAGNOSTIC ===")
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Timm:         {timm.__version__}")
print(f"Device:       {device}")
if device.type == "cuda":
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"Memory Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"Memory Free:  {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / (1024**3):.2f} GB")
print("==========================")


=== RUNTIME DIAGNOSTIC ===
Python:       3.13.15
PyTorch:      2.11.0+cu128
Timm:         1.0.28
Device:       cuda
GPU:          Tesla T4
Memory Total: 14.56 GB
Memory Free:  14.56 GB


## 2. Dataset Acquisition & 38-Class Dataset Verification

If the full 38-class dataset (~54k images) is not already present at `/content/data_38`, download it via Kaggle.
*(The 15-class subset at `/content/data/PlantVillage` is preserved and not used here)*.

In [4]:
# Optional download block for full 38-class PlantVillage dataset
# !pip install -q kaggle
!mkdir -p /content/data_38
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/data_38 --unzip
print("If dataset is already downloaded at /content/data_38 or ./data_38, proceed to locator below.")

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
Resuming from 55574528 bytes (2133811891 bytes left)...
100% 2.04G/2.04G [01:43<00:00, 20.6MB/s]

If dataset is already downloaded at /content/data_38 or ./data_38, proceed to locator below.


## 3. Dynamic 38-Class Dataset Locator & Pre-Flight Dataset Audit

In [5]:
def find_38class_dataset() -> Optional[Path]:
    """Locate directory containing the full 38-class PlantVillage dataset."""
    candidates = [
        Path("/content/data_38/plantvillage dataset/color"),
        Path("/content/data_38/plantvillage dataset/segmented"),
        Path("/content/data_38/PlantVillage"),
        Path("/content/data_38"),
        Path("./data_38/plantvillage dataset/color"),
        Path("./data_38/PlantVillage"),
        Path("./data_38"),
        Path("/content/plantvillage_full"),
        Path("/content/data/plantvillage_full")
    ]
    for p in candidates:
        if p.exists() and p.is_dir():
            subdirs = [d for d in p.iterdir() if d.is_dir()]
            # Full PlantVillage has ~38 classes including Apple, Grape, Corn, Orange, Peach, etc.
            if len(subdirs) >= 30 and any("Apple" in d.name for d in subdirs):
                return p

    # Recursive search under /content and current directory
    for root_scan in [Path("/content"), Path(".")]:
        if root_scan.exists():
            for dirpath, dirnames, _ in os.walk(root_scan):
                if len(dirnames) >= 30 and any("Apple" in d for d in dirnames) and any("Grape" in d for d in dirnames):
                    return Path(dirpath)
    return None

DATA_DIR_38 = find_38class_dataset()

if DATA_DIR_38 is None:
    print("=" * 65)
    print("❌ FULL 38-CLASS PLANTVILLAGE DATASET NOT FOUND")
    print("=" * 65)
    print("The currently active dataset at /content/data/PlantVillage contains ONLY 15 classes (20,638 images).")
    print("\nTo download the full 38-class PlantVillage dataset (~54,300 images), run:")
    print("!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/data_38 --unzip")
    print("=" * 65)
    raise FileNotFoundError("Full 38-class dataset not found. Stopping to prevent training on 15-class subset.")

print(f"✓ 38-Class Dataset Located: {DATA_DIR_38.resolve()}")

# Audit classes and image counts
class_dirs = sorted([d for d in DATA_DIR_38.iterdir() if d.is_dir()])
class_names = [d.name for d in class_dirs]
num_classes = len(class_names)

print(f"Number of class folders: {num_classes}")

audit_records = []
all_samples = []
corrupt_count = 0

for class_idx, class_dir in enumerate(tqdm(class_dirs, desc="Auditing 38 Classes")):
    cls_name = class_dir.name
    files = [f for f in class_dir.iterdir() if f.is_file() and f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    valid_count = 0

    for f in files:
        try:
            with Image.open(f) as img:
                img.verify()
                valid_count += 1
                all_samples.append((str(f), class_idx))
        except Exception:
            corrupt_count += 1

    audit_records.append({
        "Class Index": class_idx,
        "Class Name": cls_name,
        "Valid Images": valid_count
    })

audit_df = pd.DataFrame(audit_records)
total_images = audit_df["Valid Images"].sum()

print("\n" + "=" * 65)
print("38-CLASS DATASET AUDIT COMPLETE")
print("=" * 65)
print(f"Dataset Path:       {DATA_DIR_38.resolve()}")
print(f"Number of Classes:  {num_classes}")
print(f"Total Valid Images: {total_images}")
print(f"Corrupt Images:     {corrupt_count}")
print("=" * 65)

display(audit_df)

✓ 38-Class Dataset Located: /content/data_38/plantvillage dataset/color
Number of class folders: 38


Auditing 38 Classes: 100%|██████████| 38/38 [00:04<00:00,  8.48it/s]


38-CLASS DATASET AUDIT COMPLETE
Dataset Path:       /content/data_38/plantvillage dataset/color
Number of Classes:  38
Total Valid Images: 54305
Corrupt Images:     0


,Class Index,Class Name,Valid Images
0,0,Apple___Apple_scab,630
1,1,Apple___Black_rot,621
2,2,Apple___Cedar_apple_rust,275
3,3,Apple___healthy,1645
4,4,Blueberry___healthy,1502
5,5,Cherry_(including_sour)___Powdery_mildew,1052
6,6,Cherry_(including_sour)___healthy,854
7,7,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,513
8,8,Corn_(maize)___Common_rust_,1192
9,9,Corn_(maize)___Northern_Leaf_Blight,985


## 4. Reproducible Stratified Dataset Splitting (80% Train / 10% Val / 10% Test)

In [6]:
# Stratified 80/10/10 Split with seed 42
all_labels = [s[1] for s in all_samples]

train_samples, temp_samples, train_labels, temp_labels = train_test_split(
    all_samples,
    all_labels,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=all_labels
)

val_samples, test_samples, val_labels, test_labels = train_test_split(
    temp_samples,
    temp_labels,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp_labels
)

# Overlap Assertions
train_paths = set(s[0] for s in train_samples)
val_paths = set(s[0] for s in val_samples)
test_paths = set(s[0] for s in test_samples)

assert len(train_paths.intersection(val_paths)) == 0, "Overlap in Train and Val!"
assert len(train_paths.intersection(test_paths)) == 0, "Overlap in Train and Test!"
assert len(val_paths.intersection(test_paths)) == 0, "Overlap in Val and Test!"
assert len(train_samples) + len(val_samples) + len(test_samples) == len(all_samples)

print(f"--- STRATIFIED SPLIT SUMMARY (38 Classes) ---")
print(f"Total Samples: {len(all_samples)}")
print(f"Train Count:   {len(train_samples)} ({len(train_samples)/len(all_samples)*100:.1f}%)")
print(f"Val Count:     {len(val_samples)} ({len(val_samples)/len(all_samples)*100:.1f}%)")
print(f"Test Count:    {len(test_samples)} ({len(test_samples)/len(all_samples)*100:.1f}%)")

--- STRATIFIED SPLIT SUMMARY (38 Classes) ---
Total Samples: 54305
Train Count:   43444 (80.0%)
Val Count:     5430 (10.0%)
Test Count:    5431 (10.0%)


## 5. EfficientNet-B3 Preprocessing & DataLoaders (300×300)

In [7]:
IMG_SIZE = 300
BATCH_SIZE = 16  # 16 per batch on T4 prevents OOM with 300x300 images
GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch size = 32
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class SampleListDataset(Dataset):
    def __init__(self, samples: List[Tuple[str, int]], transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        path, label = self.samples[idx]
        with Image.open(path) as img:
            image = img.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

train_dataset = SampleListDataset(train_samples, transform=train_transform)
val_dataset = SampleListDataset(val_samples, transform=val_test_transform)
test_dataset = SampleListDataset(test_samples, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True if torch.cuda.is_available() else False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True if torch.cuda.is_available() else False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True if torch.cuda.is_available() else False)

print(f"DataLoaders constructed (Batch Size={BATCH_SIZE}, Effective={BATCH_SIZE*GRADIENT_ACCUMULATION_STEPS}):")
print(f" - Train Batches: {len(train_loader)} ({len(train_dataset)} samples)")
print(f" - Val Batches:   {len(val_loader)} ({len(val_dataset)} samples)")
print(f" - Test Batches:  {len(test_loader)} ({len(test_dataset)} samples)")


DataLoaders constructed (Batch Size=16, Effective=32):
 - Train Batches: 2716 (43444 samples)
 - Val Batches:   340 (5430 samples)
 - Test Batches:  340 (5431 samples)


## 6. Pre-Flight Diagnostic Check

In [8]:
# Pre-flight single batch verification
sample_imgs, sample_lbls = next(iter(train_loader))
sample_imgs, sample_lbls = sample_imgs.to(device), sample_lbls.to(device)

OUTPUT_DIR = Path("/content/exported_models_38")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("=== PRE-FLIGHT CHECK ===")
print(f"DATASET:              PlantVillage 38-Class")
print(f"CLASSES:              {num_classes}")
print(f"TOTAL IMAGES:         {len(all_samples)}")
print(f"GPU:                  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"AVAILABLE GPU MEMORY: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB" if torch.cuda.is_available() else "N/A")
print(f"OUTPUT DIRECTORY:     {OUTPUT_DIR.resolve()}")
print("=" * 60)
print("38-CLASS TRAINING READY")
print("=" * 60)

=== PRE-FLIGHT CHECK ===
DATASET:              PlantVillage 38-Class
CLASSES:              38
TOTAL IMAGES:         54305
GPU:                  Tesla T4
AVAILABLE GPU MEMORY: 14.56 GB
OUTPUT DIRECTORY:     /content/exported_models_38
38-CLASS TRAINING READY


## 7. Model Architecture, Class Weights & Mixed Precision Setup

In [9]:
def create_38class_model(num_classes: int = 38, pretrained: bool = True) -> nn.Module:
    """Build EfficientNet-B3 classification model for 38 classes."""
    model = timm.create_model("efficientnet_b3", pretrained=pretrained, num_classes=num_classes)
    return model

# Calculate balanced class weights for 38 classes
train_labels_arr = np.array(train_labels)
computed_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_labels_arr
)
class_weights_tensor = torch.tensor(computed_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
scaler = GradScaler()  # For AMP Mixed Precision training on Tesla T4

/tmp/ipykernel_1846/2229534959.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # For AMP Mixed Precision training on Tesla T4


## 8. Full Training Loop with Mixed Precision (AMP) & Early Stopping

Trains for ~15 epochs with early stopping on validation Macro F1.

In [10]:
import gc

# Clean GPU memory before training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = create_38class_model(num_classes=num_classes, pretrained=True).to(device)

optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

NUM_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4

best_model_wts = copy.deepcopy(model.state_dict())
best_val_macro_f1 = 0.0
best_epoch = 0
epochs_no_improve = 0

history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "val_balanced_acc": [],
    "val_macro_f1": [], "val_weighted_f1": [], "lr": []
}

print(f"Starting 38-Class Training: {NUM_EPOCHS} Epochs with Mixed Precision on {device}...")

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    optimizer.zero_grad()

    for batch_idx, (inputs, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{NUM_EPOCHS:02d} [Train]")):
        inputs, labels = inputs.to(device), labels.to(device)

        # Mixed precision forward pass
        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss = loss / GRADIENT_ACCUMULATION_STEPS

        if torch.isnan(loss):
            raise ValueError(f"NaN loss at epoch {epoch+1}")

        scaler.scale(loss).backward()

        # Step optimizer with gradient accumulation
        if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS * inputs.size(0)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_targets = []

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/{NUM_EPOCHS:02d} [Val]"):
            inputs, labels = inputs.to(device), labels.to(device)
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            preds = torch.argmax(outputs, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_dataset)
    val_acc = accuracy_score(val_targets, val_preds)
    val_bal_acc = balanced_accuracy_score(val_targets, val_preds)
    val_macro_f1 = f1_score(val_targets, val_preds, average="macro")
    val_weighted_f1 = f1_score(val_targets, val_preds, average="weighted")

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_macro_f1)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_balanced_acc"].append(val_bal_acc)
    history["val_macro_f1"].append(val_macro_f1)
    history["val_weighted_f1"].append(val_weighted_f1)
    history["lr"].append(current_lr)

    epoch_time = time.time() - epoch_start
    print(
        f"Epoch {epoch+1:02d}/{NUM_EPOCHS:02d} ({epoch_time:.1f}s) | "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}% BalAcc: {val_bal_acc*100:.2f}% MacroF1: {val_macro_f1:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
        print(f"  ★ New best 38-class checkpoint saved! (Val Macro F1: {best_val_macro_f1:.4f})")
    else:
        epochs_no_improve += 1
        print(f"  No improvement for {epochs_no_improve}/{EARLY_STOPPING_PATIENCE} epochs.")
        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}. Restoring best weights from epoch {best_epoch}.")
            break

model.load_state_dict(best_model_wts)
print(f"\nTraining complete. Best checkpoint was Epoch {best_epoch} with Val Macro F1: {best_val_macro_f1:.4f}")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Starting 38-Class Training: 15 Epochs with Mixed Precision on cuda...


Epoch 01/15 [Train]:   0%|          | 0/2716 [00:00<?, ?it/s]/tmp/ipykernel_1846/1692593475.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 01/15 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]/tmp/ipykernel_1846/1692593475.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 01/15 [Val]: 100%|██████████| 340/340 [00:31<00:00, 10.64it/s]


...

Epoch 13/15 (630.7s) | Train Loss: 0.0033 Acc: 99.89% | Val Loss: 0.0244 Acc: 99.67% BalAcc: 99.37% MacroF1: 0.9942 | LR: 0.000100
  No improvement for 2/4 epochs.


Epoch 14/15 [Train]:   0%|          | 0/2716 [00:00<?, ?it/s]/tmp/ipykernel_1846/1692593475.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 14/15 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]/tmp/ipykernel_1846/1692593475.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 14/15 [Val]: 100%|██████████| 340/340 [00:28<00:00, 11.94it/s]


Epoch 14/15 (624.0s) | Train Loss: 0.0064 Acc: 99.77% | Val Loss: 0.0163 Acc: 99.63% BalAcc: 99.40% MacroF1: 0.9937 | LR: 0.000100
  No improvement for 3/4 epochs.


Epoch 15/15 [Train]:   0%|          | 0/2716 [00:00<?, ?it/s]/tmp/ipykernel_1846/1692593475.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 15/15 [Val]:   0%|          | 0/340 [00:00<?, ?it/s]/tmp/ipykernel_1846/1692593475.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 15/15 [Val]: 100%|██████████| 340/340 [00:28<00:00, 11.91it/s]

Epoch 15/15 (629.3s) | Train Loss: 0.0026 Acc: 99.91% | Val Loss: 0.0131 Acc: 99.82% BalAcc: 99.71% MacroF1: 0.9967 | LR: 0.000050
  ★ New best 38-class checkpoint saved! (Val Macro F1: 0.9967)

Training complete. Best checkpoint was Epoch 15 with Val Macro F1: 0.9967


## 9. Untouched Test Set Evaluation (38 Classes)

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns

model.eval()
test_preds = []
test_targets = []
test_probs = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs, labels = inputs.to(device), labels.to(device)
        with autocast():
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

        test_probs.extend(probs.cpu().numpy())
        test_preds.extend(preds.cpu().numpy())
        test_targets.extend(labels.cpu().numpy())

test_preds_arr = np.array(test_preds)
test_targets_arr = np.array(test_targets)
test_probs_arr = np.array(test_probs)

test_acc = accuracy_score(test_targets_arr, test_preds_arr)
test_bal_acc = balanced_accuracy_score(test_targets_arr, test_preds_arr)
test_macro_f1 = f1_score(test_targets_arr, test_preds_arr, average="macro")
test_weighted_f1 = f1_score(test_targets_arr, test_preds_arr, average="weighted")
test_top3_acc = top_k_accuracy_score(test_targets_arr, test_probs_arr, k=min(3, num_classes))

report_text = classification_report(test_targets_arr, test_preds_arr, target_names=class_names)
cm = confusion_matrix(test_targets_arr, test_preds_arr)

print("========================================")
print("   38-CLASS UNTOUCHED TEST SET METRICS  ")
print("========================================")
print(f"Accuracy:          {test_acc*100:.2f}%")
print(f"Balanced Accuracy: {test_bal_acc*100:.2f}%")
print(f"Macro F1:          {test_macro_f1:.4f}")
print(f"Weighted F1:       {test_weighted_f1:.4f}")
print(f"Top-3 Accuracy:    {test_top3_acc*100:.2f}%")
print("----------------------------------------")
print("\nPer-Class Classification Report:\n")
print(report_text)

Testing:   0%|          | 0/340 [00:00<?, ?it/s]/tmp/ipykernel_1846/2463035212.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Testing: 100%|██████████| 340/340 [00:33<00:00, 10.10it/s]

   38-CLASS UNTOUCHED TEST SET METRICS  
Accuracy:          99.87%
Balanced Accuracy: 99.81%
Macro F1:          0.9975
Weighted F1:       0.9987
Top-3 Accuracy:    99.98%
----------------------------------------

Per-Class Classification Report:

                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       1.00      1.00      1.00        63
                                 Apple___Black_rot       1.00      1.00      1.00        62
                          Apple___Cedar_apple_rust       1.00      1.00      1.00        27
                                   Apple___healthy       1.00      1.00      1.00       165
                               Blueberry___healthy       1.00      1.00      1.00       150
          Cherry_(including_sour)___Powdery_mildew       1.00      1.00      1.00       105
                 Cherry_(including_sour)___healthy       1.00      0.99      0.99        85
Corn_(maize)___C

## 10. Probability Calibration & FarmFusion Safety Tiers

In [12]:
# Expected Calibration Error (ECE) & Brier Score
confidences = np.max(test_probs_arr, axis=1)
predictions = np.argmax(test_probs_arr, axis=1)
accuracies = (predictions == test_targets_arr)

n_bins = 10
bin_boundaries = np.linspace(0, 1, n_bins + 1)
ece = 0.0
for i in range(n_bins):
    in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
    prop = np.mean(in_bin)
    if prop > 0:
        ece += np.abs(np.mean(confidences[in_bin]) - np.mean(accuracies[in_bin])) * prop

one_hot = np.zeros_like(test_probs_arr)
for i, t in enumerate(test_targets_arr):
    one_hot[i, t] = 1.0
brier = np.mean(np.sum((test_probs_arr - one_hot)**2, axis=1))

# FarmFusion Safety Confidence Tier Breakdown
tier_high = np.sum(confidences >= 0.75)
tier_med = np.sum((confidences >= 0.45) & (confidences < 0.75))
tier_low = np.sum((confidences >= 0.30) & (confidences < 0.45))
tier_unclear = np.sum(confidences < 0.30)
tot = len(confidences)

print("=== 38-CLASS CALIBRATION & SAFETY METRICS ===")
print(f"Brier Score:                {brier:.4f}")
print(f"Expected Calibration Error: {ece:.4f}")
print(f"\nFarmFusion Confidence Tier Distribution:")
print(f"  - HIGH    (>= 0.75):   {tier_high:5d} ({tier_high/tot*100:.1f}%)")
print(f"  - MEDIUM  (0.45-0.74): {tier_med:5d} ({tier_med/tot*100:.1f}%)")
print(f"  - LOW     (0.30-0.44): {tier_low:5d} ({tier_low/tot*100:.1f}%)")
print(f"  - UNCLEAR (< 0.30):    {tier_unclear:5d} ({tier_unclear/tot*100:.1f}%)")
print("=============================================")

=== 38-CLASS CALIBRATION & SAFETY METRICS ===
Brier Score:                0.0022
Expected Calibration Error: 0.0008

FarmFusion Confidence Tier Distribution:
  - HIGH    (>= 0.75):    5428 (99.9%)
  - MEDIUM  (0.45-0.74):     3 (0.1%)
  - LOW     (0.30-0.44):     0 (0.0%)
  - UNCLEAR (< 0.30):        0 (0.0%)


## 11. Export Production Artifacts to `/content/exported_models_38/`

In [13]:
EXPORT_DIR_38 = Path("/content/exported_models_38")
EXPORT_DIR_38.mkdir(parents=True, exist_ok=True)

# 1. Weights
model_path = EXPORT_DIR_38 / "disease_model_v2_38class.pth"
torch.save(model.state_dict(), model_path)

# 2. Label mapping
mapping_path = EXPORT_DIR_38 / "disease_label_mapping_v2_38class.json"
mapping_data = {
    "num_classes": num_classes,
    "class_names": class_names,
    "class_to_idx": {name: i for i, name in enumerate(class_names)},
    "idx_to_class": {str(i): name for i, name in enumerate(class_names)}
}
with open(mapping_path, "w") as f:
    json.dump(mapping_data, f, indent=2)

# 3. Model metadata
metadata_path = EXPORT_DIR_38 / "disease_model_metadata_v2_38class.json"
metadata_data = {
    "model_name": "FarmFusion Disease Model V2 (38-Class)",
    "architecture": "efficientnet_b3",
    "image_size": IMG_SIZE,
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    },
    "dataset_source": "PlantVillage Full Dataset (38 Classes)",
    "num_classes": num_classes,
    "class_names": class_names,
    "splits": {
        "total_images": len(all_samples),
        "train_count": len(train_samples),
        "val_count": len(val_samples),
        "test_count": len(test_samples),
        "random_seed": RANDOM_SEED
    },
    "training": {
        "epochs_trained": len(history["train_loss"]),
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "optimizer": "AdamW",
        "learning_rate": 2e-4,
        "precision": "Mixed Precision (AMP FP16)"
    },
    "test_metrics": {
        "accuracy": float(test_acc),
        "balanced_accuracy": float(test_bal_acc),
        "macro_f1": float(test_macro_f1),
        "weighted_f1": float(test_weighted_f1),
        "top3_accuracy": float(test_top3_acc),
        "brier_score": float(brier),
        "ece": float(ece)
    },
    "confidence_thresholds": {
        "HIGH": ">=0.75",
        "MEDIUM": "0.45-0.74",
        "LOW": "0.30-0.44",
        "UNCLEAR": "<0.30"
    },
    "timestamp_utc": datetime.datetime.utcnow().isoformat(),
    "framework_versions": {
        "torch": torch.__version__,
        "timm": timm.__version__
    }
}

with open(metadata_path, "w") as f:
    json.dump(metadata_data, f, indent=2)

print("Export verified:")
for p in [model_path, mapping_path, metadata_path]:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f" ✓ {p.name:36} ({size_mb:.2f} MB)")

Export verified:
 ✓ disease_model_v2_38class.pth         (41.56 MB)
 ✓ disease_label_mapping_v2_38class.json (0.00 MB)
 ✓ disease_model_metadata_v2_38class.json (0.00 MB)


/tmp/ipykernel_1846/3947559001.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.datetime.utcnow().isoformat(),


## 12. Final Training Report

In [14]:
print("""
========================================
FARMFUSION 38-CLASS DISEASE MODEL V2
========================================

Dataset: PlantVillage (Full 38-Class)
Images:  {total_images}
Classes: 38
GPU:     {gpu_name}

Training:
Train:       {train_count} images
Validation:  {val_count} images
Test:        {test_count} images

Best Epoch:               {best_epoch}
Best Validation Macro F1: {best_val_macro_f1:.4f}

TEST RESULTS
Accuracy:          {test_acc:.2f}%
Balanced Accuracy: {bal_acc:.2f}%
Macro F1:          {macro_f1:.4f}
Weighted F1:       {weighted_f1:.4f}
Top-3 Accuracy:    {top3_acc:.2f}%

Artifacts:
- /content/exported_models_38/disease_model_v2_38class.pth
- /content/exported_models_38/disease_label_mapping_v2_38class.json
- /content/exported_models_38/disease_model_metadata_v2_38class.json
========================================
""".format(
    total_images=len(all_samples),
    gpu_name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    train_count=len(train_samples),
    val_count=len(val_samples),
    test_count=len(test_samples),
    best_epoch=best_epoch,
    best_val_macro_f1=best_val_macro_f1,
    test_acc=test_acc * 100,
    bal_acc=test_bal_acc * 100,
    macro_f1=test_macro_f1,
    weighted_f1=test_weighted_f1,
    top3_acc=test_top3_acc * 100
))


FARMFUSION 38-CLASS DISEASE MODEL V2

Dataset: PlantVillage (Full 38-Class)
Images:  54305
Classes: 38
GPU:     Tesla T4

Training:
Train:       43444 images
Validation:  5430 images
Test:        5431 images

Best Epoch:               15
Best Validation Macro F1: 0.9967

TEST RESULTS
Accuracy:          99.87%
Balanced Accuracy: 99.81%
Macro F1:          0.9975
Weighted F1:       0.9987
Top-3 Accuracy:    99.98%

Artifacts:
- /content/exported_models_38/disease_model_v2_38class.pth
- /content/exported_models_38/disease_label_mapping_v2_38class.json
- /content/exported_models_38/disease_model_metadata_v2_38class.json



In [15]:
from pathlib import Path

EXPORT_DIR = Path("/content/exported_models_38")

print("Exists:", EXPORT_DIR.exists())

if EXPORT_DIR.exists():
    for f in EXPORT_DIR.iterdir():
        if f.is_file():
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"✓ {f.name:45} {size_mb:.2f} MB")
else:
    print("❌ Export directory not found")

Exists: True
✓ disease_model_v2_38class.pth                  41.56 MB
✓ disease_model_metadata_v2_38class.json        0.00 MB
✓ disease_label_mapping_v2_38class.json         0.00 MB


In [16]:
from google.colab import files

files.download(
    "/content/exported_models_38/disease_model_v2_38class.pth"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
files.download(
    "/content/exported_models_38/disease_label_mapping_v2_38class.json"
)

files.download(
    "/content/exported_models_38/disease_model_metadata_v2_38class.json"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    

CUDA: True
GPU: Tesla T4


In [5]:
from pathlib import Path

print("Searching for 38-class model...")

matches = list(Path("/content").rglob("disease_model_v2_38class.pth"))

if matches:
    print("\nFOUND:")
    for p in matches:
        print(p)
else:
    print("\n❌ Model not found anywhere in /content")

Searching for 38-class model...

❌ Model not found anywhere in /content


In [6]:
from pathlib import Path

pth_files = list(Path("/content").rglob("*.pth"))

print(f"Found {len(pth_files)} .pth files:")

for p in pth_files:
    size = p.stat().st_size / (1024 * 1024)
    print(f"{p}  ({size:.2f} MB)")

Found 0 .pth files:


In [7]:
from pathlib import Path

print("=== SEARCHING PERSISTENT LOCATIONS ===")

search_locations = [
    Path("/content"),
    Path("/home"),
    Path("/mnt/data"),
]

for location in search_locations:
    if not location.exists():
        continue

    print(f"\nSearching: {location}")

    matches = list(location.rglob("disease_model_v2_38class.pth"))

    for f in matches:
        print("FOUND MODEL:", f)

    if not matches:
        print("No 38-class model found.")

=== SEARCHING PERSISTENT LOCATIONS ===

Searching: /content
No 38-class model found.

Searching: /home
No 38-class model found.


In [8]:
from google.colab import files

files.download("/content/exported_models_38/disease_model_v2_38class.pth")
files.download("/content/exported_models_38/disease_label_mapping_v2_38class.json")
files.download("/content/exported_models_38/disease_model_metadata_v2_38class.json")

FileNotFoundError: Cannot find file: /content/exported_models_38/disease_model_v2_38class.pth

In [9]:
from pathlib import Path

for base in [Path("/content"), Path("/content/exported_models_38")]:
    print("\nChecking:", base)
    if base.exists():
        for f in base.rglob("*"):
            if f.is_file():
                print(f, f.stat().st_size / (1024 * 1024), "MB")


Checking: /content
/content/.config/gce 4.76837158203125e-06 MB
/content/.config/default_configs.db 0.01171875 MB
/content/.config/active_config 6.67572021484375e-06 MB
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db 0.01171875 MB
/content/.config/.last_opt_in_prompt.yaml 2.86102294921875e-06 MB
/content/.config/.last_survey_prompt.yaml 3.528594970703125e-05 MB
/content/.config/.last_update_check.json 0.00012874603271484375 MB
/content/.config/config_sentinel 0.0 MB
/content/sample_data/anscombe.json 0.0016183853149414062 MB
/content/sample_data/README.md 0.0009174346923828125 MB
/content/sample_data/mnist_train_small.csv 34.831886291503906 MB
/content/sample_data/mnist_test.csv 17.442172050476074 MB
/content/sample_data/california_housing_train.csv 1.6273784637451172 MB
/content/sample_data/california_housing_test.csv 0.28719043731689453 MB
/content/.config/configurations/config_default 8.96453857421875e-05 MB
/content/.config/logs/2026.08.20/13.35.23.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [12]:
from pathlib import Path

matches = list(
    Path("/content/drive/MyDrive").rglob("disease_model_v2_38class.pth")
)

print("Found:", len(matches))

for p in matches:
    print(p)
    print("Size:", p.stat().st_size / (1024 * 1024), "MB")

Found: 0


In [18]:
from pathlib import Path
import zipfile

EXPORT_DIR = Path("/content/exported_models_38")
ZIP_PATH = Path("/content/FarmFusion_Disease_Model_V2_38Class.zip")

print("Checking:", EXPORT_DIR)
print("Exists:", EXPORT_DIR.exists())

if not EXPORT_DIR.exists():
    raise FileNotFoundError(f"{EXPORT_DIR} does not exist")

files_to_zip = list(EXPORT_DIR.iterdir())

print("\nFiles found:")
for f in files_to_zip:
    if f.is_file():
        print(f"  ✓ {f.name} ({f.stat().st_size / (1024*1024):.2f} MB)")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files_to_zip:
        if f.is_file():
            z.write(f, arcname=f.name)

print("\nZIP CREATED:")
print(ZIP_PATH)
print(f"Size: {ZIP_PATH.stat().st_size / (1024*1024):.2f} MB")

Checking: /content/exported_models_38
Exists: True

Files found:
  ✓ disease_model_v2_38class.pth (41.56 MB)
  ✓ disease_model_metadata_v2_38class.json (0.00 MB)
  ✓ disease_label_mapping_v2_38class.json (0.00 MB)

ZIP CREATED:
/content/FarmFusion_Disease_Model_V2_38Class.zip
Size: 38.45 MB


In [19]:
from google.colab import files

files.download("/content/FarmFusion_Disease_Model_V2_38Class.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
from pathlib import Path

zip_path = Path("/content/FarmFusion_Disease_Model_V2_38Class.zip")

print("Exists:", zip_path.exists())

if zip_path.exists():
    print("Size:", zip_path.stat().st_size / (1024 * 1024), "MB")
    print("Path:", zip_path)

Exists: True
Size: 38.4536771774292 MB
Path: /content/FarmFusion_Disease_Model_V2_38Class.zip


In [21]:
import shutil
from pathlib import Path

source = Path("/content/FarmFusion_Disease_Model_V2_38Class.zip")

destination = Path(
    "/home/rdj/FarmFusionFinal/FarmFusion_Disease_Model_V2_38Class.zip"
)

shutil.copy2(source, destination)

print("✅ Copy completed")
print("Location:", destination)
print("Size:", destination.stat().st_size / (1024 * 1024), "MB")

FileNotFoundError: [Errno 2] No such file or directory: '/home/rdj/FarmFusionFinal/FarmFusion_Disease_Model_V2_38Class.zip'

In [22]:
import os

print(os.path.exists("/content/FarmFusion_Disease_Model_V2_38Class.zip"))
print(os.path.getsize("/content/FarmFusion_Disease_Model_V2_38Class.zip") / (1024 * 1024), "MB")

True
38.4536771774292 MB


In [23]:
from pathlib import Path

drive = Path("/content/drive")

print("Drive exists:", drive.exists())

if drive.exists():
    print("Contents:")
    for x in drive.iterdir():
        print(x)

Drive exists: False


In [25]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
from pathlib import Path

drive_dir = Path("/content/drive/MyDrive/FarmFusion_Disease_Model_V2_38Class")
drive_dir.mkdir(parents=True, exist_ok=True)

print("Drive folder:", drive_dir)

Drive folder: /content/drive/MyDrive/FarmFusion_Disease_Model_V2_38Class


In [27]:
import shutil

source = Path("/content/FarmFusion_Disease_Model_V2_38Class.zip")
destination = drive_dir / source.name

shutil.copy2(source, destination)

print("✅ Model ZIP backed up to Google Drive")
print(destination)
print(f"Size: {destination.stat().st_size / (1024*1024):.2f} MB")

✅ Model ZIP backed up to Google Drive
/content/drive/MyDrive/FarmFusion_Disease_Model_V2_38Class/FarmFusion_Disease_Model_V2_38Class.zip
Size: 38.45 MB


In [28]:
import shutil

model_dir = Path("/content/exported_models_38")

for file in model_dir.iterdir():
    if file.is_file():
        shutil.copy2(file, drive_dir / file.name)
        print("✓ Backed up:", file.name)

✓ Backed up: disease_model_v2_38class.pth
✓ Backed up: disease_model_metadata_v2_38class.json
✓ Backed up: disease_label_mapping_v2_38class.json
